#  Data Profiling, Cleaning & EDA - Operational Analyst
Notebook ini melakukan pemuatan dataset, basic profiling, pembersihan data (terutama datetime & status), serta EDA operasional secara end-to-end.
Karena menggunakan notebook, kita bisa memanfaatkan fungsi `display()` bawaan Jupyter untuk merender DataFrame secara rapi.

In [ ]:
from FP import DustiniaDelixia_Groceria
from FP import DustiniaDelixia_Groceria
from FP import DustiniaDelixia_Groceria
from FP import DustiniaDelixia_Groceria
from FP import DustiniaDelixia_Groceria
from FP import DustiniaDelixia_Groceria
from FP import DustiniaDelixia_Groceria
import pandas as pd
import numpy as np
import warnings
import pandas as pd
import numpy as np
import warnings
import sys
from pathlib import Path
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

def profile_dataset(df, dataset_name):
    print("\n" + "="*80)
    print(f"📊 PROFILING DATASET: {dataset_name.upper()}")
    print("="*80)
    
    print(f"\n📐 SHAPE DATA:")
    print(f"   Rows: {df.shape[0]:,}")
    print(f"   Columns: {df.shape[1]}")
    
    missing = df.isnull().sum()
    missing_pct = (missing / len(df)) * 100
    missing_df = pd.DataFrame({
        'Missing_Count': missing,
        'Missing_Percentage': missing_pct.round(2)
    })
    missing_df = missing_df[missing_df['Missing_Count'] > 0]
    
    print(f"\n MISSING VALUES:")
    if len(missing_df) > 0:
        display(missing_df)
    else:
        print("   ✅ Tidak ada missing values!")
        
    duplicates = df.duplicated().sum()
    print(f"\n DUPLICATE ROWS: {duplicates:,} ({ (duplicates/len(df))*100 :.2f}%)")
    return df

## 1. Load Data & Basic Profiling
Memuat ketujuh dataset utama dan menjalankan basic profiling pada masing-masing dataset.

In [ ]:
datasets = {
    'orders.csv': 'Orders',
    'order_items.csv': 'Order Items',
    'sellers.csv': 'Sellers',
    'customers.csv': 'Customers',
    'geolocation.csv': 'Geolocation',
    'products.csv': 'Products',
    'order_reviews.csv': 'Order Reviews'
}

all_dfs = {}

for file_path, dataset_name in datasets.items():
    try:
        df = pd.read_csv(file_path)
        profile_dataset(df, dataset_name)
        all_dfs[dataset_name] = df
    except Exception as e:
        print(f"ERROR load {file_path}: {e}")

orders = all_dfs.get('Orders')
items = all_dfs.get('Order Items')
sellers = all_dfs.get('Sellers')
customers = all_dfs.get('Customers')
geolocation = all_dfs.get('Geolocation')
products = all_dfs.get('Products')
order_reviews = all_dfs.get('Order Reviews')

## 2. Data Cleaning Khusus Orders (Timestamp & Status)
Mengubah kolom waktu ke dalam tipe data `datetime` dan hanya memfilter pesanan dengan status 'delivered' karena kita ingin menganalisis SLA (waktu penyelesaian) akhir.

In [ ]:
if orders is not None:
    timestamp_cols = [
        'order_purchase_timestamp',
        'order_approved_at',
        'order_delivered_carrier_date',
        'order_delivered_customer_date',
        'order_estimated_delivery_date'
    ]
    
    for col in timestamp_cols:
        if col in orders.columns:
            orders[col] = pd.to_datetime(orders[col])
            
    print("✅ Kolom timestamp berhasil dikonversi ke datetime.")
    
    orders_delivered = orders[orders['order_status'] == 'delivered'].copy()
    
    initial_count = len(orders_delivered)
    orders_delivered.dropna(subset=['order_delivered_customer_date', 'order_delivered_carrier_date'], inplace=True)
    final_count = len(orders_delivered)
    print(f"✅ Menyaring pesanan sukses. Baris berkurang dari {initial_count:,} menjadi {final_count:,} (Dibuang: {initial_count - final_count:,} karena timestamp tidak lengkap).")

## 3. SLA & Lead Time Operasional
Menghitung durasi processing seller, transit kurir, total delivery time, dan tingkat keterlambatan (Delay Rate).

In [ ]:
if 'orders_delivered' in locals():
    # A. Seller Processing Time
    orders_delivered['seller_processing_days'] = (orders_delivered['order_delivered_carrier_date'] - orders_delivered['order_purchase_timestamp']).dt.total_seconds() / (24 * 3600)
    
    # B. Carrier Transit Time
    orders_delivered['carrier_transit_days'] = (orders_delivered['order_delivered_customer_date'] - orders_delivered['order_delivered_carrier_date']).dt.total_seconds() / (24 * 3600)
    
    # C. Actual Delivery Time
    orders_delivered['actual_delivery_days'] = (orders_delivered['order_delivered_customer_date'] - orders_delivered['order_purchase_timestamp']).dt.total_seconds() / (24 * 3600)
    
    # Penanganan Outlier
    initial_outlier_count = len(orders_delivered)
    orders_delivered = orders_delivered[orders_delivered['actual_delivery_days'] >= 0]
    if len(orders_delivered) < initial_outlier_count:
        print(f"✅ Membuang {initial_outlier_count - len(orders_delivered):,} baris anomali (waktu pengiriman negatif).")
        
    # D. Deviasi Estimasi vs Realitas (Delay Days)
    orders_delivered['delay_days'] = (orders_delivered['order_delivered_customer_date'] - orders_delivered['order_estimated_delivery_date']).dt.total_seconds() / (24 * 3600)
    orders_delivered['is_delayed'] = orders_delivered['delay_days'] > 0
    
    stats_cols = ['seller_processing_days', 'carrier_transit_days', 'actual_delivery_days', 'delay_days']
    print("\n--- STATISTIK DESKRIPTIF SLA OPERASIONAL (Hari) ---")
    display(orders_delivered[stats_cols].describe().round(2))
    
    delay_rate = orders_delivered['is_delayed'].mean() * 100
    print(f"\n🚨 TINGKAT KETERLAMBATAN TOTAL (Delay Rate): {delay_rate:.2f}%")

## 4. Analisis Rute Geografis & Bottleneck
Menggabungkan data orders dengan items, sellers, dan customers untuk melihat rute mana yang paling lama/sering terlambat.

In [ ]:
if all(v is not None for v in [items, sellers, customers]) and 'orders_delivered' in locals():
    merged_df = pd.merge(items, orders_delivered, on='order_id', how='inner')
    merged_df = pd.merge(merged_df, sellers, on='seller_id', how='inner')
    merged_df = pd.merge(merged_df, customers, on='customer_id', how='inner')
    
    route_perf = merged_df.groupby(['seller_state', 'customer_state']).agg(
        total_transactions=('order_id', 'nunique'),
        avg_delivery_days=('actual_delivery_days', 'mean'),
        avg_delay_days=('delay_days', 'mean'),
        delay_rate=('is_delayed', 'mean')
    ).reset_index()
    
    route_perf = route_perf[route_perf['total_transactions'] > 50]
    route_perf['delay_rate_pct'] = (route_perf['delay_rate'] * 100).round(2)
    
    print("✈️ 5 RUTE PENGIRIMAN DENGAN RATA-RATA WAKTU TERLAMA (Hari):")
    display(route_perf.sort_values(by='avg_delivery_days', ascending=False).head(5))
    
    print("\n🚨 5 RUTE PENGIRIMAN DENGAN TINGKAT DELAY PALING PARAH (%):")
    display(route_perf.sort_values(by='delay_rate_pct', ascending=False).head(5)[['seller_state', 'customer_state', 'total_transactions', 'delay_rate_pct', 'avg_delivery_days']])

## 5. EDA Tambahan: Geolocation, Products & Reviews
Analisis fitur lain seperti koordinat, terjemahan kategori produk, dan dampak delay pengiriman terhadap kepuasan pelanggan (nilai review).

In [ ]:
# A. GEOLOCATION
if geolocation is not None:
    print("\n--- GEOLOCATION EDA ---")
    geo = geolocation.copy()
    geo_count_before = len(geo)
    geo.drop_duplicates(inplace=True)
    print(f"✅ Duplikat dihapus. Sisa baris: {len(geo):,}")
    display(geo['geolocation_city'].value_counts().head(5).to_frame("Top 5 Cities"))

# B. PRODUCTS
if products is not None:
    print("\n--- PRODUCTS EDA ---")
    prod = products.copy()
    prod['product_category_name'] = prod['product_category_name'].fillna('unknown')
    
    try:
        translation = pd.read_csv('category_translation.csv')
        prod = pd.merge(prod, translation, on='product_category_name', how='left')
        prod['product_category_name_english'] = prod['product_category_name_english'].fillna(prod['product_category_name'])
        cat_col = 'product_category_name_english'
        print("✅ Berhasil translasi kategori ke bahasa Inggris")
    except:
        cat_col = 'product_category_name'
        
    display(prod[cat_col].value_counts().head(5).to_frame("Top 5 Product Categories"))

# C. ORDER REVIEWS
if order_reviews is not None and 'orders_delivered' in locals():
    print("\n--- ORDER REVIEWS EDA ---")
    rev = order_reviews.copy()
    rev['review_creation_date'] = pd.to_datetime(rev['review_creation_date'])
    
    print(f"Rata-rata Review Score: {rev['review_score'].mean():.2f} / 5.0")
    
    if 'delay_days' in orders_delivered.columns:
        print("\n🔍 Korelasi Keterlambatan vs Review Score:")
        order_rev = pd.merge(orders_delivered[['order_id', 'is_delayed']], rev[['order_id', 'review_score']], on='order_id', how='inner')
        delay_impact = order_rev.groupby('is_delayed')['review_score'].mean().round(2)
        delay_impact.index = delay_impact.index.map({False: 'Tepat Waktu / Lebih Cepat', True: 'Terlambat'})
        display(delay_impact.to_frame("Average Review Score"))